In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from pathlib import Path
import sys
import os
from sklearn.preprocessing import LabelEncoder


project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.utils.nbaPlayerLogs import NBAGameLogs
from src.features.feature_engineer.pbp_features import *
from src.features.feature_engineer.pbp_min_features import *

pd.set_option('display.max_columns', None)

In [2]:
s22 = pd.read_csv(project_root / 'data/raw/pbp_season_stats/S22.csv')
s23 = pd.read_csv(project_root / 'data/raw/pbp_season_stats/S23.csv')
s24 = pd.read_csv(project_root / 'data/raw/pbp_season_stats/S24.csv')
s25 = pd.read_csv(project_root / 'data/raw/pbp_season_stats/S25.csv')
s26 = pd.read_csv(project_root / 'data/raw/pbp_season_stats/S26.csv')
s26.head()

,gameId,actionNumber,clock,period,teamId,teamTricode,personId,playerName,playerNameI,xLegacy,yLegacy,shotDistance,shotResult,isFieldGoal,scoreHome,scoreAway,pointsTotal,location,description,actionType,subType,videoAvailable,shotValue,actionId
0,22500086,2,PT12M00.00S,1,0,NaN,0,NaN,NaN,0,0,0,NaN,0,0.0,0.0,0,NaN,Start of 1st Period (8:14 PM EST),period,start,1,0,1
1,22500086,4,PT12M00.00S,1,1610612749,MIL,1626167,Turner,M. Turner,0,0,0,NaN,0,NaN,NaN,0,h,Jump Ball Turner vs. Sarr: Tip to McCollum,Jump Ball,NaN,1,0,2
2,22500086,7,PT11M50.00S,1,1610612749,MIL,1631260,Green,A. Green,0,0,0,NaN,0,NaN,NaN,0,h,Green P.FOUL (P1.T1) (B.Forte),Foul,Personal,1,0,3
3,22500086,9,PT11M42.00S,1,1610612764,WAS,203114,Middleton,K. Middleton,130,-5,13,Missed,1,NaN,NaN,0,v,MISS Middleton 13' Turnaround Fadeaway Shot,Missed Shot,Turnaround Fadeaway shot,1,2,4
4,22500086,10,PT11M38.00S,1,1610612764,WAS,1642259,Sarr,A. Sarr,0,0,0,NaN,0,NaN,NaN,0,v,Sarr REBOUND (Off:1 Def:0),Rebound,Unknown,1,0,5


In [3]:
game_ids = s23['gameId'].unique()[:10]

res = []
for game_id in game_ids:
    print(f"Analyzing game: {game_id}")
    game_df = s23[s23['gameId'] == game_id].copy()
    game_df = cleanPlaybyPlay(game_df)
    game_df = add_lineup_columns(game_df)
    game_df = analyze_player_minutes(game_df)
    by_game = game_df["by_game"]
    by_game["gameId"] = game_id
    res.append((by_game))
final_df = pd.concat(res, ignore_index=True)
final_df.sample(5)

Analyzing game: 22200007

⚠  1 substitution(s) skipped — lineup held at last known state:
   Period 2 [home] 'SUB: Highsmith FOR Adebayo' | out=1628389 | 'Highsmith' not in name lookup for team 1610612748
  Stints extracted  : 83
  Players tracked   : 18
  Periods covered   : [np.int64(1), np.int64(2), np.int64(3), np.int64(4)]
Analyzing game: 22200002
✓ All substitutions resolved. Every row has exactly 5 per team.
  Stints extracted  : 100
  Players tracked   : 25
  Periods covered   : [np.int64(1), np.int64(2), np.int64(3), np.int64(4)]
Analyzing game: 22200011
✓ All substitutions resolved. Every row has exactly 5 per team.
  Stints extracted  : 91
  Players tracked   : 28
  Periods covered   : [np.int64(1), np.int64(2), np.int64(3), np.int64(4)]
Analyzing game: 22200012

⚠  4 substitution(s) skipped — lineup held at last known state:
   Period 4 [away] 'SUB: Nnaji FOR Gordon' | out=203932 | 'Nnaji' not in name lookup for team 1610612743
   Period 4 [home] 'SUB: Fontecchio FOR Sexton

,personId,playerName,teamId,total_game_min,total_stints,avg_stint_min,max_stint_min,periods_played,periods_started,first_period_on,avg_entry_sec,entry_regularity_std,first_stint_min,in_closing_lineup,clutch_min,clutch_min_rate,gameId
8,203083,A. Drummond,1610612741,14.88,4,3.72,5.23,4,2,1,435.75,328.27,2.63,False,0.0000,0.00,22200007
10,200768,K. Lowry,1610612748,34.65,4,8.66,9.58,4,2,1,595.50,147.42,9.37,True,0.0000,0.00,22200007
38,1629308,J. Toscano-Anderson,1610612747,14.30,5,2.86,5.40,4,1,1,363.00,207.05,4.15,True,0.0000,0.00,22200002
153,201988,P. Mills,1610612751,21.33,5,4.27,6.38,4,1,1,367.94,252.34,6.03,False,0.0000,0.00,22200006
126,1628969,M. Bridges,1610612756,35.80,4,8.95,12.00,4,3,1,651.75,136.50,12.00,True,4.6667,0.13,22200013


In [25]:
s23_df = pd.read_csv(project_root / 'data/raw/season_stats/S23.csv')
s23_df.head()

,Unnamed: 0.3,Unnamed: 0.2,Unnamed: 0.1,Unnamed: 0,SEASON_YEAR,PLAYER_ID,PLAYER_NAME,NICKNAME,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,TOV,STL,BLK,BLKA,PF,PFD,PTS,PLUS_MINUS,NBA_FANTASY_PTS,DD2,TD3,WNBA_FANTASY_PTS,AVAILABLE_FLAG,MIN_SEC,TEAM_COUNT,E_OFF_RATING,OFF_RATING,sp_work_OFF_RATING,E_DEF_RATING,DEF_RATING,sp_work_DEF_RATING,E_NET_RATING,NET_RATING,sp_work_NET_RATING,AST_PCT,AST_TO,AST_RATIO,OREB_PCT,DREB_PCT,REB_PCT,TM_TOV_PCT,E_TOV_PCT,EFG_PCT,TS_PCT,USG_PCT,E_USG_PCT,E_PACE,PACE,PACE_PER40,sp_work_PACE,PIE,POSS,FGM_PG,FGA_PG,START_POSITION,SPD,DIST,ORBC,DRBC,RBC,TCHS,SAST,FTAST,PASS,CFGM,CFGA,CFG_PCT,UFGM,UFGA,UFG_PCT,DFGM,DFGA,DFG_PCT,TEAM_FGM,TEAM_FGA,TEAM_FG_PCT,TEAM_FG3M,TEAM_FG3A,TEAM_FG3_PCT,TEAM_FTM,TEAM_FTA,TEAM_FT_PCT,TEAM_OREB,TEAM_DREB,TEAM_REB,TEAM_AST,TEAM_TOV,TEAM_STL,TEAM_BLK,TEAM_BLKA,TEAM_PF,TEAM_PFD,TEAM_PTS,TEAM_PLUS_MINUS,TEAM_E_OFF_RATING,TEAM_OFF_RATING,TEAM_E_DEF_RATING,TEAM_DEF_RATING,TEAM_E_NET_RATING,TEAM_NET_RATING,TEAM_AST_PCT,TEAM_AST_TO,TEAM_AST_RATIO,TEAM_OREB_PCT,TEAM_DREB_PCT,TEAM_REB_PCT,TEAM_TM_TOV_PCT,TEAM_EFG_PCT,TEAM_TS_PCT,TEAM_E_PACE,TEAM_PACE,TEAM_PACE_PER40,TEAM_POSS,TEAM_PIE,OPP_TEAM_ID,OPP_OPP_ABBREVIATION_base,OPP_OPP_NAME_base,OPP_FGM,OPP_FGA,OPP_FG_PCT,OPP_FG3M,OPP_FG3A,OPP_FG3_PCT,OPP_FTM,OPP_FTA,OPP_FT_PCT,OPP_OREB,OPP_DREB,OPP_REB,OPP_AST,OPP_TOV,OPP_STL,OPP_BLK,OPP_BLKA,OPP_PF,OPP_PFD,OPP_PTS,OPP_PLUS_MINUS,OPP_E_OFF_RATING,OPP_OFF_RATING,OPP_E_DEF_RATING,OPP_DEF_RATING,OPP_E_NET_RATING,OPP_NET_RATING,OPP_AST_PCT,OPP_AST_TO,OPP_AST_RATIO,OPP_OREB_PCT,OPP_DREB_PCT,OPP_REB_PCT,OPP_TM_TOV_PCT,OPP_EFG_PCT,OPP_TS_PCT,OPP_E_PACE,OPP_PACE,OPP_PACE_PER40,OPP_POSS,OPP_PIE,IS_PLAYOFF,POS,AGE,TEAM_SPREAD_ODDS,GAME_TOTAL_ODDS,TEAM_SPREAD,GAME_TOTAL,STARTING,PTS_PER_MIN,AST_PER_MIN,REB_PER_MIN,IS_HOME,POSITION_ENCODED
0,0,0,0,25893,2022-23,1629680,Matisse Thybulle,Matisse,1610612755,PHI,Philadelphia 76ers,22200001,2022-10-18,PHI @ BOS,L,0.403333,0,0,0.000,0,0,0.000,0,0,0.000,0,0,0,0,0,0,0,0,0,0,0,-1,0.0,0,0,0.0,1,0:24,1,0.0,0.0,0.0,53.2,50.0,50.0,-53.2,-50.0,-50.0,0.000,0.00,0.0,0.000,0.000,0.000,0.0,0.0,0.000,0.000,0.000,0.000,111.87,178.51,148.76,178.51,0.000,1,0.0,0.0,NaN,4.53,0.03,0,0,0,0,0,0,0,0,0,0.000,0,0,0.000,0,0,0.0,40,80,0.500,13,34,0.382,24,28,0.857,4,27,31,16,14.0,8,3,3,25,24,117,-9.0,114.3,119.4,126.9,129.9,-12.5,-10.5,0.400,1.14,13.1,0.190,0.744,0.457,0.143,0.581,0.634,100.8,97.5,81.25,98,0.434,1610612738,BOS,Boston Celtics,46,82,0.561,12,35,0.343,22,28,0.786,6,30,36,24,11.0,8,3,3,24,25,126,9.0,126.9,129.9,114.3,119.4,12.5,10.5,0.522,2.18,18.5,0.256,0.810,0.543,0.113,0.634,0.668,100.8,97.5,81.25,97,0.566,0,SG,25.0,3.0,216.0,3.0,216.5,0,0.000000,0.000000,0.000000,0,4
1,25,1,25,25867,2022-23,203210,JaMychal Green,JaMychal,1610612744,GSW,Golden State Warriors,22200002,2022-10-18,GSW vs. LAL,W,23.433333,3,6,0.500,2,3,0.667,0,0,0.000,5,2,7,0,0,1,0,0,1,0,8,-3,19.4,0,0,19.0,1,23:26,1,89.3,91.1,91.1,97.5,98.2,98.2,-8.1,-7.1,-7.1,0.000,0.00,0.0,0.152,0.077,0.119,0.0,0.0,0.667,0.667,0.087,0.091,115.20,113.68,94.74,113.68,0.117,56,3.0,6.0,NaN,4.13,1.63,5,5,10,29,0,0,23,1,3,0.333,2,3,0.667,2,4,0.5,45,99,0.455,16,45,0.356,17,23,0.739,11,37,48,31,18.0,11,4,4,23,18,123,14.0,105.9,107.0,92.4,97.3,13.6,9.6,0.689,1.72,19.1,0.281,0.754,0.518,0.157,0.535,0.564,117.1,113.5,94.58,115,0.548,1610612747,LAL,Los Angeles Lakers,40,94,0.426,10,40,0.250,19,25,0.760,9,39,48,23,22.0,12,4,4,18,23,109,-14.0,92.4,97.3,105.9,107.0,-13.6,-9.6,0.575,1.05,15.4,0.246,0.719,0.482,0.196,0.479,0.519,117.1,113.5,94.58,112,0.452,0,PF,32.0,-7.5,223.5,-7.5,223.5,0,0.341394,0.000000,0.298720,1,1
2,26,2,26,25866,2022-23,1629684,Grant Williams,Grant,1610612738,BOS,Boston Celtics,22200001,2022-10-18,BOS vs. PHI,W,23.950000,5,5,1.000,3,3,1.000,2,3,0.667,0,1,1,1,0,0,0,0,3,2,15,-3,17.7,0,0,20.0,1,23:57,1,110.1,114.6,114.6,113.6,118.4,118.4,-3.5,-3.8,-3.8,0.063,0.00,14.3,0.000,0.048,0.024,0.0,0.0,1.300,1.187,0.111,0.117,101.

In [26]:
final_df.rename(columns={"personId": "PLAYER_ID","gameId": "GAME_ID"}, inplace=True)
s23_df = s23_df.merge(final_df, on=["GAME_ID", "PLAYER_ID"], how="left")
s23_df.head()

,Unnamed: 0.3,Unnamed: 0.2,Unnamed: 0.1,Unnamed: 0,SEASON_YEAR,PLAYER_ID,PLAYER_NAME,NICKNAME,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,TOV,STL,BLK,BLKA,PF,PFD,PTS,PLUS_MINUS,NBA_FANTASY_PTS,DD2,TD3,WNBA_FANTASY_PTS,AVAILABLE_FLAG,MIN_SEC,TEAM_COUNT,E_OFF_RATING,OFF_RATING,sp_work_OFF_RATING,E_DEF_RATING,DEF_RATING,sp_work_DEF_RATING,E_NET_RATING,NET_RATING,sp_work_NET_RATING,AST_PCT,AST_TO,AST_RATIO,OREB_PCT,DREB_PCT,REB_PCT,TM_TOV_PCT,E_TOV_PCT,EFG_PCT,TS_PCT,USG_PCT,E_USG_PCT,E_PACE,PACE,PACE_PER40,sp_work_PACE,PIE,POSS,FGM_PG,FGA_PG,START_POSITION,SPD,DIST,ORBC,DRBC,RBC,TCHS,SAST,FTAST,PASS,CFGM,CFGA,CFG_PCT,UFGM,UFGA,UFG_PCT,DFGM,DFGA,DFG_PCT,TEAM_FGM,TEAM_FGA,TEAM_FG_PCT,TEAM_FG3M,TEAM_FG3A,TEAM_FG3_PCT,TEAM_FTM,TEAM_FTA,TEAM_FT_PCT,TEAM_OREB,TEAM_DREB,TEAM_REB,TEAM_AST,TEAM_TOV,TEAM_STL,TEAM_BLK,TEAM_BLKA,TEAM_PF,TEAM_PFD,TEAM_PTS,TEAM_PLUS_MINUS,TEAM_E_OFF_RATING,TEAM_OFF_RATING,TEAM_E_DEF_RATING,TEAM_DEF_RATING,TEAM_E_NET_RATING,TEAM_NET_RATING,TEAM_AST_PCT,TEAM_AST_TO,TEAM_AST_RATIO,TEAM_OREB_PCT,TEAM_DREB_PCT,TEAM_REB_PCT,TEAM_TM_TOV_PCT,TEAM_EFG_PCT,TEAM_TS_PCT,TEAM_E_PACE,TEAM_PACE,TEAM_PACE_PER40,TEAM_POSS,TEAM_PIE,OPP_TEAM_ID,OPP_OPP_ABBREVIATION_base,OPP_OPP_NAME_base,OPP_FGM,OPP_FGA,OPP_FG_PCT,OPP_FG3M,OPP_FG3A,OPP_FG3_PCT,OPP_FTM,OPP_FTA,OPP_FT_PCT,OPP_OREB,OPP_DREB,OPP_REB,OPP_AST,OPP_TOV,OPP_STL,OPP_BLK,OPP_BLKA,OPP_PF,OPP_PFD,OPP_PTS,OPP_PLUS_MINUS,OPP_E_OFF_RATING,OPP_OFF_RATING,OPP_E_DEF_RATING,OPP_DEF_RATING,OPP_E_NET_RATING,OPP_NET_RATING,OPP_AST_PCT,OPP_AST_TO,OPP_AST_RATIO,OPP_OREB_PCT,OPP_DREB_PCT,OPP_REB_PCT,OPP_TM_TOV_PCT,OPP_EFG_PCT,OPP_TS_PCT,OPP_E_PACE,OPP_PACE,OPP_PACE_PER40,OPP_POSS,OPP_PIE,IS_PLAYOFF,POS,AGE,TEAM_SPREAD_ODDS,GAME_TOTAL_ODDS,TEAM_SPREAD,GAME_TOTAL,STARTING,PTS_PER_MIN,AST_PER_MIN,REB_PER_MIN,IS_HOME,POSITION_ENCODED,playerName,teamId,total_game_min,total_stints,avg_stint_min,max_stint_min,periods_played,periods_started,first_period_on,avg_entry_sec,entry_regularity_std
0,0,0,0,25893,2022-23,1629680,Matisse Thybulle,Matisse,1610612755,PHI,Philadelphia 76ers,22200001,2022-10-18,PHI @ BOS,L,0.403333,0,0,0.000,0,0,0.000,0,0,0.000,0,0,0,0,0,0,0,0,0,0,0,-1,0.0,0,0,0.0,1,0:24,1,0.0,0.0,0.0,53.2,50.0,50.0,-53.2,-50.0,-50.0,0.000,0.00,0.0,0.000,0.000,0.000,0.0,0.0,0.000,0.000,0.000,0.000,111.87,178.51,148.76,178.51,0.000,1,0.0,0.0,NaN,4.53,0.03,0,0,0,0,0,0,0,0,0,0.000,0,0,0.000,0,0,0.0,40,80,0.500,13,34,0.382,24,28,0.857,4,27,31,16,14.0,8,3,3,25,24,117,-9.0,114.3,119.4,126.9,129.9,-12.5,-10.5,0.400,1.14,13.1,0.190,0.744,0.457,0.143,0.581,0.634,100.8,97.5,81.25,98,0.434,1610612738,BOS,Boston Celtics,46,82,0.561,12,35,0.343,22,28,0.786,6,30,36,24,11.0,8,3,3,24,25,126,9.0,126.9,129.9,114.3,119.4,12.5,10.5,0.522,2.18,18.5,0.256,0.810,0.543,0.113,0.634,0.668,100.8,97.5,81.25,97,0.566,0,SG,25.0,3.0,216.0,3.0,216.5,0,0.000000,0.000000,0.000000,0,4,M. Thybulle,1.610613e+09,0.40,2.0,0.20,0.34,2.0,0.0,1.0,12.70,12.45
1,25,1,25,25867,2022-23,203210,JaMychal Green,JaMychal,1610612744,GSW,Golden State Warriors,22200002,2022-10-18,GSW vs. LAL,W,23.433333,3,6,0.500,2,3,0.667,0,0,0.000,5,2,7,0,0,1,0,0,1,0,8,-3,19.4,0,0,19.0,1,23:26,1,89.3,91.1,91.1,97.5,98.2,98.2,-8.1,-7.1,-7.1,0.000,0.00,0.0,0.152,0.077,0.119,0.0,0.0,0.667,0.667,0.087,0.091,115.20,113.68,94.74,113.68,0.117,56,3.0,6.0,NaN,4.13,1.63,5,5,10,29,0,0,23,1,3,0.333,2,3,0.667,2,4,0.5,45,99,0.455,16,45,0.356,17,23,0.739,11,37,48,31,18.0,11,4,4,23,18,123,14.0,105.9,107.0,92.4,97.3,13.6,9.6,0.689,1.72,19.1,0.281,0.754,0.518,0.157,0.535,0.564,117.1,113.5,94.58,115,0.548,1610612747,LAL,Los Angeles Lakers,40,94,0.426,10,40,0.250,19,25,0.760,9,39,48,23,22.0,12,4,4,18,23,109,-14.0,92.4,97.3,105.9,107.0,-13.6,-9.6,0.575,1.05,15.4,0.246,0.719,0.482,0.196,0.479,0.519,117.1,113.5,94.58,112,0.452,0,PF,32.0,-7.5,223.5,-7.5,223.5,0,0.341394,0.000000,0.298720,1,1,J. Green,1.610613e+09,23.43,4.0,5.86,10.13,4.0,2.0,1.0,535.00,213.78
2,26,2,26,25866,2022-23,1629684,Gra

In [27]:
s23_df.to_csv(project_root / 'data/raw/season_stats/s23_pbp.csv', index=False)